# Validate a credit-risk score with Toetra

This notebook shows the workflow an ML engineer could use before deployment:

1. train and serialize a small affine scoring model;
2. define behavioral requirements in a `.toetra` policy;
3. run `verify(...)` without manipulating IR2 or Z3 directly;
4. inspect the rich HTML report;
5. replay a counterexample on the original sklearn model;
6. export stable JSON and standalone HTML artifacts.

The example uses **transformed numerical features** (`debt_ratio` and `late_payments`). Encoding preprocessing pipelines is deliberately outside the current V1 scope.

In [ ]:
from pathlib import Path
import sys

# Source-checkout bootstrap. Installed Toetra users do not need this block.
_repository_candidates = (Path.cwd(), *Path.cwd().parents)
REPOSITORY_ROOT = next(
    (
        candidate
        for candidate in _repository_candidates
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "toetra").is_dir()
    ),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError(
        "Unable to locate the Toetra repository root from the notebook directory."
    )

_source_root_text = str(REPOSITORY_ROOT / "src")
if _source_root_text not in sys.path:
    sys.path.insert(0, _source_root_text)

REPOSITORY_ROOT


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression

from toetra import verify
from toetra.examples import credit_risk_policy

POLICY = credit_risk_policy()
workspace_handle = TemporaryDirectory(prefix="toetra-credit-risk-notebook-")
WORKSPACE = Path(workspace_handle.name)
WORKSPACE


## 1. Build a small reference model

The synthetic target is intentionally affine so that the current Z3 profile can encode it:

\[
\text{risk\_score} = 0.6 \times \text{debt\_ratio}
                  + 0.08 \times \text{late\_payments}
                  + 0.1
\]

The score is illustrative rather than a calibrated probability.

In [ ]:
rows = []
for debt_ratio in (0.0, 0.25, 0.5, 0.75, 1.0):
    for late_payments in (0.0, 1.0, 2.0, 3.0, 4.0, 5.0):
        rows.append(
            {
                "debt_ratio": debt_ratio,
                "late_payments": late_payments,
                "risk_score": 0.6 * debt_ratio + 0.08 * late_payments + 0.1,
            }
        )

reference = pd.DataFrame(rows)
features = ["debt_ratio", "late_payments"]
model = LinearRegression().fit(reference[features], reference["risk_score"])

model_path = WORKSPACE / "credit_risk.joblib"
dataset_path = WORKSPACE / "credit_risk_reference.csv"
joblib.dump(model, model_path)
reference.to_csv(dataset_path, index=False)

pd.DataFrame(
    {
        "feature": features,
        "coefficient": model.coef_,
    }
).assign(intercept=model.intercept_)

## 2. Review the behavioral policy

The policy proves a global upper bound, deliberately exposes a stricter rule that has a counterexample, and asks for an applicant reaching the manual-review threshold.

In [ ]:
print(POLICY)


## 3. Verify the serialized model

`verify(...)` performs model introspection, semantic validation, IR construction, capability routing, Z3 execution and report construction. In Jupyter, returning the session as the final expression invokes its `_repr_html_()` method.

In [ ]:
session = verify(
    POLICY,
    model=model_path,
    dataset=dataset_path,
)
session


In [ ]:
session.to_dataframe()


## 4. Replay the counterexample on sklearn

Toetra keeps the original estimator attached to artifact-based sessions. The first counterexample can therefore be replayed without parsing solver values or reconstructing feature names manually.


In [ ]:
counterexample = session.first_counterexample
assert counterexample is not None

replay = counterexample.replay()
replay.to_dataframe()


## 5. Export review artifacts

The JSON contract is versioned for CI or downstream automation. The HTML file is self-contained and can be attached to a model-review ticket without requiring Jupyter.

In [ ]:
artifacts = session.write_artifacts(
    WORKSPACE / "reports",
    formats={"json", "html"},
)

{
    **artifacts,
    "ci_exit_code": session.exit_code,
}


A non-zero `session.exit_code` is suitable for a CI gate. This notebook intentionally contains a counterexample, so the overall session exits with code `1` even though that counterexample is expected for demonstration purposes.